# NS punctuality summary

Builds a yearly, per-service-type punctuality summary from the [rijdendetreinen.nl](https://www.rijdendetreinen.nl/) open data (raw `services-YYYY.csv` files), and exports it to Excel.

Only services operated by **NS** (`Service:Company == "NS"`) are included — NS Int., Thalys/Eurostar-style international services running on other companies' codes are excluded upstream by that filter (see `data_NS_exploratory_analysis.ipynb` for how that was determined).

**Output columns** (one row per year x service type):

| Column | Meaning |
|---|---|
| Year | Calendar year, from the source filename |
| Service type | `Service:Type` as reported by NS (Intercity, Sprinter, replacement bus/metro services, ...) |
| Total trains scheduled | Distinct services (`Service:RDT-ID`) planned |
| Total trains fully cancelled | Distinct services fully cancelled (`Service:Completely cancelled`) |
| Total trains partially cancelled | Distinct services partially, not fully, cancelled (`Service:Partly cancelled`) — some stop along the route was skipped, but not necessarily the terminus |
| Total fully + partially | Sum of the two above (a service is never counted in both) |
| Total trains arriving at terminus | Distinct services whose **terminus stop itself** was not cancelled (`Stop:Arrival cancelled == False` on the last stop) — see method notes, this is *not* simply `scheduled - cancelled` |
| Trains on time at terminus (<=3/5/10/15 min) | Among the trains arriving at terminus, count whose **arrival delay at the terminus** was within the threshold |

### Method notes

- **Terminus** = the last stop recorded for a service. The source CSV lists a service's stops together and in order, so the last row seen for a given `Service:RDT-ID` is its terminus — there is no explicit stop-sequence column to sort by (verified in the exploratory notebook).
- **A partially cancelled service can still reach its terminus.** `Service:Partly cancelled` just means *some* stop on the route was skipped — it does not say which one. Checked directly against 2023 data: of the services flagged partly (not fully) cancelled, 50.7% still had a non-cancelled arrival at their terminus stop. So "arrived at terminus" is computed from the **stop-level** `Stop:Arrival cancelled` flag on the terminus row itself, not from the service-level `Completely`/`Partly cancelled` flags — those two only answer "was any part of this service cancelled," not "did it reach the end."
- **Cancellation de-duplication**: a service can be flagged both `Completely cancelled` and `Partly cancelled` at once. It is counted as *fully* cancelled in that case, never both, so `Total fully + partially` never double-counts a service (same convention as the exploratory notebook). This only affects the two cancellation columns, not `Total trains arriving at terminus`.
- **On-time thresholds** use `Stop:Arrival delay <= threshold` (minutes) at the terminus, restricted to services whose terminus arrival was not cancelled. A missing arrival delay at the terminus counts as *not* on time.
- The files are ~3 GB each, so they are streamed in chunks rather than loaded whole. A service's rows can straddle a chunk boundary; the last (possibly incomplete) service in each chunk is held over and prepended to the next chunk before aggregating, so every service is only aggregated once its rows are all in hand.


In [ ]:
import re
import time
from pathlib import Path

import pandas as pd


## Config


In [ ]:
# Raw data: one services-YYYY.csv per year, from rijdendetreinen.nl
DATA_DIR = Path(
    r"D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A"
    r"\Shared knowledge tools and data\data_sources\Rail_databases"
    r"\punctuality_big_data\NL_rijdendetreinen"
)
FILENAME_RE = re.compile(r"^services-(\d{4})\.csv$")

OUTPUT_PATH = Path("output") / "NS_punctuality_summary.xlsx"

OPERATORS = ["NS"]  # Service:Company values to keep (excludes NS Int., other operators)
ON_TIME_THRESHOLDS = [3, 5, 10, 15]  # minutes
CHUNKSIZE = 2_000_000  # rows per chunk while streaming each ~3 GB file

USECOLS = [
    "Service:RDT-ID",
    "Service:Type",
    "Service:Company",
    "Service:Completely cancelled",
    "Service:Partly cancelled",
    "Stop:Arrival delay",
    "Stop:Arrival cancelled",
]
DTYPES = {
    "Service:RDT-ID": "int64",
    "Service:Type": "category",
    "Service:Company": "category",
}


## Per-file aggregation

`process_year_file` streams one `services-YYYY.csv` in chunks and returns a `{service_type: stats}` dict. `process_complete` is only ever called with rows whose services are fully contained in the batch it receives (chunk-boundary services are held over in `pending` by the caller).


In [ ]:
def _new_stat():
    stat = {"scheduled": 0, "fully_cancelled": 0, "partly_cancelled": 0, "arrived": 0}
    for thr in ON_TIME_THRESHOLDS:
        stat[f"on_time_{thr}"] = 0
    return stat


def process_complete(df, stats):
    """Aggregate a batch of rows whose services are fully contained in df."""
    if df.empty:
        return

    per_service = df.groupby("Service:RDT-ID", sort=False, observed=True).agg(
        **{
            "Service:Type": ("Service:Type", "first"),
            "Service:Completely cancelled": ("Service:Completely cancelled", "first"),
            "Service:Partly cancelled": ("Service:Partly cancelled", "first"),
        }
    )
    for svc_type, sub in per_service.groupby("Service:Type", observed=True):
        stat = stats.setdefault(svc_type, _new_stat())
        fully = sub["Service:Completely cancelled"]
        partly_only = (~fully) & sub["Service:Partly cancelled"]
        stat["scheduled"] += len(sub)
        stat["fully_cancelled"] += int(fully.sum())
        stat["partly_cancelled"] += int(partly_only.sum())

    # Terminus = last row seen per service (rows for a service are contiguous, see notes above).
    # Whether the service reached it is read off the terminus row\'s own Stop:Arrival cancelled
    # flag, not the service-level Completely/Partly cancelled flags - a partly cancelled service
    # can still have reached its terminus (some other stop was the one skipped).
    terminus = df[~df["Service:RDT-ID"].duplicated(keep="last")]
    arrived = terminus[terminus["Stop:Arrival cancelled"] == False]
    for svc_type, sub in arrived.groupby("Service:Type", observed=True):
        stat = stats.setdefault(svc_type, _new_stat())
        stat["arrived"] += len(sub)
        delay = sub["Stop:Arrival delay"]
        for thr in ON_TIME_THRESHOLDS:
            stat[f"on_time_{thr}"] += int((delay <= thr).sum())


def process_year_file(path, chunksize=CHUNKSIZE, verbose=True):
    stats = {}
    pending = None
    t0 = time.time()
    n_rows = 0
    for chunk in pd.read_csv(
        path,
        usecols=USECOLS,
        dtype=DTYPES,
        chunksize=chunksize,
        true_values=["true"],
        false_values=["false"],
    ):
        n_rows += len(chunk)
        chunk = chunk[chunk["Service:Company"].isin(OPERATORS)]
        if pending is not None:
            chunk = pd.concat([pending, chunk], ignore_index=True)
        if chunk.empty:
            pending = None
            continue
        last_id = chunk["Service:RDT-ID"].iloc[-1]
        is_pending = chunk["Service:RDT-ID"] == last_id
        process_complete(chunk[~is_pending], stats)
        pending = chunk[is_pending]
        if verbose:
            print(f"    {n_rows:,} rows read ({time.time() - t0:.0f}s)", end="\r")
    if pending is not None:
        process_complete(pending, stats)
    if verbose:
        print(f"    {n_rows:,} rows read ({time.time() - t0:.0f}s) - done")
    return stats


## Discover available years


In [ ]:
year_files = {}
for f in DATA_DIR.glob("services-*.csv"):
    m = FILENAME_RE.match(f.name)
    if m:
        year_files[int(m.group(1))] = f

year_files = dict(sorted(year_files.items()))
for year, f in year_files.items():
    print(year, "->", f.name, f"({f.stat().st_size / 1e9:.1f} GB)")


## Process every year

This reads all discovered files end to end (each ~3 GB) and can take a few minutes in total.


In [ ]:
all_stats = {}
for year, path in year_files.items():
    print(f"{year}: {path.name}")
    all_stats[year] = process_year_file(path)


## Build the summary table


In [ ]:
rows = []
for year, stats in all_stats.items():
    for svc_type, stat in stats.items():
        row = {
            "Year": year,
            "Service type": svc_type,
            "Total trains scheduled": stat["scheduled"],
            "Total trains fully cancelled": stat["fully_cancelled"],
            "Total trains partially cancelled": stat["partly_cancelled"],
            "Total fully + partially": stat["fully_cancelled"] + stat["partly_cancelled"],
            "Total trains arriving at terminus": stat["arrived"],
        }
        for thr in ON_TIME_THRESHOLDS:
            row[f"Trains on time at terminus (<= {thr} min)"] = stat[f"on_time_{thr}"]
        rows.append(row)

summary = pd.DataFrame(rows).sort_values(["Year", "Service type"]).reset_index(drop=True)
summary


## Export to Excel


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="NS punctuality", index=False)
    ws = writer.sheets["NS punctuality"]
    for col_cells in ws.columns:
        width = max(len(str(c.value)) for c in col_cells) + 2
        ws.column_dimensions[col_cells[0].column_letter].width = width

print(f"Wrote {len(summary)} rows to {OUTPUT_PATH.resolve()}")
